In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.metrics import precision_recall_curve

# ==========================================
# 1. PERSIAPAN DATA
# ==========================================
print("Memuat dataset...")
# Pastikan Anda sudah mengunduh file creditcard.csv dari Kaggle
df = pd.read_csv('creditcard1.csv')

# Menangani missing values (jika ada) sesuai instruksi jurnal
if df.isnull().sum().max() > 0:
    df = df.fillna(df.mean())

# Memisahkan fitur (X) dan target (y)
# Kelas 0: Sah, Kelas 1: Penipuan
X = df.drop(['Class', 'Time'], axis=1) # Time biasanya diabaikan dalam model ini
y = df['Class']

# Z-score Normalization (Standarisasi)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# ==========================================
# 2. DEFINISI MODEL AUTOENCODER (RBCE)
# ==========================================
def build_ae_rbce_model(input_dim, lambda_l2=0.01):
    """
    Membangun arsitektur Autoencoder.
    RBCE diimplementasikan dengan menggabungkan Binary Crossentropy
    dan L2 Regularization pada bobot jaringan.
    """
    input_layer = Input(shape=(input_dim,))
    
    # Encoder (512 neuron, ReLU, L2 Regularizer)
    encoded = Dense(512, activation='relu', kernel_regularizer=l2(lambda_l2))(input_layer)
    
    # Latent Space
    latent_space = Dense(256, activation='relu', kernel_regularizer=l2(lambda_l2))(encoded)
    
    # Decoder (512 neuron, ReLU, L2 Regularizer)
    decoded = Dense(512, activation='relu', kernel_regularizer=l2(lambda_l2))(latent_space)
    
    # Output Layer
    output_layer = Dense(input_dim, activation='sigmoid')(decoded)
    
    autoencoder = Model(inputs=input_layer, outputs=output_layer)
    
    # Optimizer Adam dengan learning rate 0.0001
    optimizer = Adam(learning_rate=0.0001)
    
    # Fungsi kerugian Binary Crossentropy (ditambah L2 dari kernel_regularizer menjadi RBCE)
    autoencoder.compile(optimizer=optimizer, loss='binary_crossentropy')
    
    return autoencoder

# ==========================================
# 3. CROSS VALIDATION & PELATIHAN
# ==========================================
# Stratified K-Fold dengan K=5
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
smote = SMOTE(random_state=42)

fold_no = 1
acc_scores, dr_scores, auc_scores = [], [], []

for train_index, test_index in skf.split(X_scaled, y):
    print(f"\n--- Melatih Fold {fold_no} ---")
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Terapkan SMOTE pada data latih
    print("Menerapkan SMOTE...")
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    
    # Inisialisasi model
    ae_model = build_ae_rbce_model(X_train_res.shape[1], lambda_l2=0.001)
    
    # Pelatihan Model (Batch size 64)
    print("Memulai proses training...")
    ae_model.fit(
        X_train_res, X_train_res, # Autoencoder belajar merekonstruksi inputnya sendiri
        epochs=50, # Diset 10 agar tidak terlalu lama saat uji coba awal, bisa dinaikkan nanti
        batch_size=64, 
        shuffle=True,
        validation_split=0.1,
        verbose=1
    )
    
    # ==========================================
    # 4. EVALUASI
    # ==========================================
    # Melakukan prediksi (rekonstruksi) pada data uji
    X_test_predictions = ae_model.predict(X_test)
    
    X_test_predictions = ae_model.predict(X_test)
    mse = np.mean(np.power(X_test - X_test_predictions, 2), axis=1)
    
    # 1. Mencari Threshold Optimal secara Dinamis berbasis F1-Score
    precisions, recalls, thresholds = precision_recall_curve(y_test, mse)
    
    # Hindari pembagian dengan nol
    f1_scores = np.divide(2 * (precisions * recalls), (precisions + recalls), 
                          out=np.zeros_like(precisions), where=(precisions + recalls) != 0)
    
    # Ambil index dari F1-score tertinggi
    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx]
    
    print(f"--> Threshold optimal yang ditemukan: {optimal_threshold:.5f}")
    
    # 2. Klasifikasi menggunakan threshold optimal tersebut
    y_pred = (mse > optimal_threshold).astype(int)
    
    # 3. Menghitung Metrik Evaluasi
    acc = accuracy_score(y_test, y_pred)
    dr = recall_score(y_test, y_pred) # Detection Rate
    auc = roc_auc_score(y_test, mse)
    
    acc_scores.append(acc)
    dr_scores.append(dr)
    auc_scores.append(auc)
    
    print(f"Hasil Fold {fold_no}:")
    print(f"Accuracy: {acc * 100:.3f}%")
    print(f"Detection Rate: {dr * 100:.3f}%")
    print(f"AUC: {auc * 100:.3f}%")

Memuat dataset...

--- Melatih Fold 1 ---
Menerapkan SMOTE...
Memulai proses training...
Epoch 1/50
6398/6398 ━━━━━━━━━━━━━━━━━━━━ 31s 4ms/step - loss: 0.6340 - val_loss: 0.6003
Epoch 2/50
6398/6398 ━━━━━━━━━━━━━━━━━━━━ 27s 4ms/step - loss: 0.5962 - val_loss: 0.5998
Epoch 3/50
6398/6398 ━━━━━━━━━━━━━━━━━━━━ 27s 4ms/step - loss: 0.5959 - val_loss: 0.5995
Epoch 4/50
6398/6398 ━━━━━━━━━━━━━━━━━━━━ 29s 4ms/step - loss: 0.5957 - val_loss: 0.5994
Epoch 5/50
6398/6398 ━━━━━━━━━━━━━━━━━━━━ 29s 5ms/step - loss: 0.5956 - val_loss: 0.5992
Epoch 6/50
6398/6398 ━━━━━━━━━━━━━━━━━━━━ 28s 4ms/step - loss: 0.5955 - val_loss: 0.5992
Epoch 7/50
6398/6398 ━━━━━━━━━━━━━━━━━━━━ 27s 4ms/step - loss: 0.5954 - val_loss: 0.5991
Epoch 8/50
6398/6398 ━━━━━━━━━━━━━━━━━━━━ 27s 4ms/step - loss: 0.5954 - val_loss: 0.5990
Epoch 9/50
6398/6398 ━━━━━━━━━━━━━━━━━━━━ 25s 4ms/step - loss: 0.5953 - val_loss: 0.5990
Epoch 10/50
6398/6398 ━━━━━━━━━━━━━━━━━━━━ 25s 4ms/step - loss: 0.5953 - val_loss: 0.5989
Epoch 11/50
6398/639

KeyboardInterrupt: 

**PREPROCESSING**